Copyright (C) 2026 S.H.Tekur

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import os
import copy
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torchvision import models

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps"  if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)


In [ ]:
class SmoothedBCEWithLogitsLoss(nn.Module):
    """
    BCEWithLogitsLoss with mild label smoothing.
    smoothing=0.05 only prevents overconfidence;
    kept low so the chirality signal stays sharp.
    """
    def __init__(self, smoothing=0.05, pos_weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        targets_smooth = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets_smooth)


def binary_accuracy(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    return (preds == targets).float().mean().item()


class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.0):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter   = 0

    def step(self, current_loss):
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter   = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, total_samples = 0.0, 0.0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_acc     += binary_accuracy(logits.detach(), labels) * bs
        total_samples += bs

    return total_loss / total_samples, total_acc / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, total_samples = 0.0, 0.0, 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(images)
        loss   = criterion(logits, labels)

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_acc     += binary_accuracy(logits, labels) * bs
        total_samples += bs

        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs,  dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return total_loss / total_samples, total_acc / total_samples, all_probs, all_labels


In [ ]:
class RealSpaceRSDataset(Dataset):
    """
    Binary R/S classifier for real-space rendered images.
    Label 0 = R, Label 1 = S.
    Physics: single mirror (hflip OR vflip) inverts chirality → label flips.
             double mirror = 180° rotation (proper rotation) → label unchanged.
    """
    def __init__(self, root_dir, split='train', image_size=224):
        self.data_dir    = Path(root_dir) / split
        self.image_paths = []
        self.labels      = []
        self.is_train    = (split == 'train')
        self.image_size  = image_size

        for class_dir in sorted(self.data_dir.iterdir()):
            if class_dir.is_dir() and class_dir.name in ['R', 'S']:
                label = 0 if class_dir.name == 'R' else 1
                for img_path in sorted(class_dir.glob('*.*')):
                    if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                        self.image_paths.append(img_path)
                        self.labels.append(label)

        if self.is_train:
            self.base_transforms = transforms.Compose([
                transforms.RandomResizedCrop(
                    size=(image_size, image_size),
                    scale=(0.80, 1.0),
                    ratio=(0.90, 1.10),
                    interpolation=InterpolationMode.BILINEAR,
                    antialias=True
                ),
                transforms.ToTensor(),
            ])
        else:
            self.base_transforms = transforms.Compose([
                transforms.Resize((image_size, image_size), antialias=True),
                transforms.ToTensor(),
            ])

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
        # Small random erasing: occludes patches to prevent spatial memorisation
        self.random_erasing = transforms.RandomErasing(
            p=0.3, scale=(0.02, 0.08), ratio=(0.5, 2.0), value=1.0
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        img = self.base_transforms(img)

        if self.is_train:
            # ── Colour jitter: prevents shortcutting on Avogadro render colour ──
            if random.random() < 0.4:
                img = transforms.ColorJitter(
                    brightness=0.2, contrast=0.2,
                    saturation=0.1, hue=0.05
                )(img)

            # ── Affine geometry ───────────────────────────────────────────────
            angle     = random.uniform(-45, 45)
            max_d     = int(0.02 * self.image_size)
            translate = [random.randint(-max_d, max_d),
                         random.randint(-max_d, max_d)]
            scale     = random.uniform(0.97, 1.03)

            img = TF.affine(
                img, angle=angle, translate=translate, scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=[1.0, 1.0, 1.0]
            )

            # ── Physics-aware flips ───────────────────────────────────────────
            # Single mirror  → chirality inverts → label flips
            # Double mirror  → 180° rotation     → chirality preserved, net flip = 0
            hflip_done = random.random() < 0.5
            vflip_done = random.random() < 0.5

            if hflip_done:
                img   = TF.hflip(img)
                label = 1 - label

            if vflip_done:
                img   = TF.vflip(img)
                label = 1 - label

        img = self.normalize(img)

        if self.is_train:
            img = self.random_erasing(img)

        return img, torch.tensor(label, dtype=torch.float32)


In [ ]:
ROOT_DIR    = "realspace"
IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_WORKERS = 0

train_rs = RealSpaceRSDataset(ROOT_DIR, "train", IMAGE_SIZE)
val_rs   = RealSpaceRSDataset(ROOT_DIR, "val",   IMAGE_SIZE)
test_rs  = RealSpaceRSDataset(ROOT_DIR, "test",  IMAGE_SIZE)

# ── Class balance check ──────────────────────────────────────────────────────
for name, ds in [("Train", train_rs), ("Val", val_rs), ("Test", test_rs)]:
    c = Counter(ds.labels)
    print(f"{name}: {len(ds)} total | R={c[0]}  S={c[1]}")

n_neg = Counter(train_rs.labels)[0]
n_pos = Counter(train_rs.labels)[1]
pos_weight_value  = n_neg / n_pos if n_pos > 0 else 1.0
pos_weight_tensor = torch.tensor([pos_weight_value], device=device)
print(f"\npos_weight = {pos_weight_value:.3f}  (1.0 = balanced)")

# ── Seeded DataLoaders ───────────────────────────────────────────────────────
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

train_loader_rs = DataLoader(
    train_rs, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g
)
val_loader_rs  = DataLoader(val_rs,  batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS)
test_loader_rs = DataLoader(test_rs, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
def build_model(freeze_backbone=False, dropout_p=0.3):
    """
    dropout_p=0.3: light enough to let the chirality gradient through,
    strong enough to prevent head co-adaptation.
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(model.fc.in_features, 1)
    )
    return model


In [ ]:
set_seed(42)

NUM_EPOCHS_S1 = 10
NUM_EPOCHS_S2 = 40
WEIGHT_DECAY  = 5e-4
PATIENCE      = 12
MODEL_PATH_RS = "real_space_rs_classifier.pth"

criterion_rs = SmoothedBCEWithLogitsLoss(
    smoothing=0.05,               # mild — keeps the chirality signal sharp
    pos_weight=pos_weight_tensor
)

model_rs = build_model(freeze_backbone=True, dropout_p=0.3).to(device)

optimizer_s1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_rs.parameters()),
    lr=1e-4, weight_decay=WEIGHT_DECAY
)
scheduler_s1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s1, mode='min', patience=3, factor=0.5
)
early_stopper_s1 = EarlyStopping(patience=5, min_delta=1e-4)

best_val_loss = float("inf")
best_state    = copy.deepcopy(model_rs.state_dict())

print("=" * 65)
print("STAGE 1 — Warm-up: head only (backbone frozen)")
print("=" * 65)

for epoch in range(NUM_EPOCHS_S1):
    train_loss, train_acc = train_one_epoch(
        model_rs, train_loader_rs, criterion_rs, optimizer_s1, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model_rs, val_loader_rs, criterion_rs, device
    )
    scheduler_s1.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS_S1} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = copy.deepcopy(model_rs.state_dict())
        torch.save(best_state, MODEL_PATH_RS)
        print(f"  ↳ Saved (val_loss={best_val_loss:.4f})")

    if early_stopper_s1.step(val_loss):
        print(f"Early stopping at epoch {epoch+1}")
        break

model_rs.load_state_dict(best_state)
print(f"\nStage 1 done. Best val loss: {best_val_loss:.4f}")


In [ ]:
set_seed(42)

# ── Unfreeze everything ──────────────────────────────────────────────────────
for param in model_rs.parameters():
    param.requires_grad = True

# 3-tier LR groups:
#   early layers  (layer1+layer2): 5e-6  — barely move; protect low-level features
#   late layers   (layer3+layer4): 5e-5  — moderate adaptation to chirality cues
#   head          (fc):            1e-4  — normal; freshest weights, needs most update
early_layers = (list(model_rs.layer1.parameters())
              + list(model_rs.layer2.parameters()))
late_layers  = (list(model_rs.layer3.parameters())
              + list(model_rs.layer4.parameters()))
head_params  =  list(model_rs.fc.parameters())

optimizer_s2 = torch.optim.AdamW([
    {'params': early_layers, 'lr': 5e-6,  'weight_decay': WEIGHT_DECAY},
    {'params': late_layers,  'lr': 5e-5,  'weight_decay': WEIGHT_DECAY},
    {'params': head_params,  'lr': 1e-4,  'weight_decay': WEIGHT_DECAY},
])

# ReduceLROnPlateau: cuts LR when val_loss stalls, safe for this dataset size
scheduler_s2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s2, mode='min', patience=5, factor=0.5, verbose=True
)
early_stopper_s2 = EarlyStopping(patience=PATIENCE, min_delta=1e-4)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

g2 = torch.Generator()
g2.manual_seed(42)
train_loader_s2 = DataLoader(
    train_rs, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g2
)

print("=" * 65)
print("STAGE 2 — Full fine-tune, 3-tier LRs, no Mixup")
print("=" * 65)
print(f"  early LR=5e-6 | late LR=5e-5 | head LR=1e-4")
print("=" * 65)

for epoch in range(NUM_EPOCHS_S2):
    train_loss, train_acc = train_one_epoch(
        model_rs, train_loader_s2, criterion_rs, optimizer_s2, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model_rs, val_loader_rs, criterion_rs, device
    )
    scheduler_s2.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS_S2} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = copy.deepcopy(model_rs.state_dict())
        torch.save(best_state, MODEL_PATH_RS)
        print(f"  ↳ Saved best model (val_loss={best_val_loss:.4f})")

    if early_stopper_s2.step(val_loss):
        print(f"Early stopping at epoch {epoch+1}")
        break

model_rs.load_state_dict(best_state)
print(f"\nStage 2 done. Best val loss: {best_val_loss:.4f}")
